# 🧱 Création de la base de données PostgreSQL pour MovieLens

## 📌 Objectif
Ce notebook a pour but de connecter notre projet Python à une base de données PostgreSQL locale nommée `movielens`, puis d’y insérer les données du dataset MovieLens Small.

Nous allons :
- Connecter Python à PostgreSQL via SQLAlchemy
- Insérer les tables à partir des fichiers CSV (`movies`, `ratings`, `tags`, `links`)
- Vérifier les données importées

> Cette base de données sera utilisée pour des requêtes SQL et pour alimenter Power BI.


In [24]:
# Connexion à la base PostgreSQL locale
from sqlalchemy import create_engine
import pandas as pd

# 🔐 Paramètres de connexion à adapter
user = "postgres"
password = "dadoo"  
host = "localhost"
port = "5432"
database = "movielens"

# 🔌 Création de l'engine SQLAlchemy
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")

# Test rapide de connexion
try:
    with engine.connect() as connection:
        print("✅ Connexion réussie à PostgreSQL.")
except Exception as e:
    print("❌ Échec de la connexion :", e)


✅ Connexion réussie à PostgreSQL.


In [25]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS ratings CASCADE;
        DROP TABLE IF EXISTS tags CASCADE;
        DROP TABLE IF EXISTS links CASCADE;
        DROP TABLE IF EXISTS movies CASCADE;
    """))
print("🗑️ Toutes les tables ont été supprimées de la base PostgreSQL.")


🗑️ Toutes les tables ont été supprimées de la base PostgreSQL.


## 🧱 Création manuelle des tables PostgreSQL avec types explicites

Nous recréons ici les tables avec un schéma défini manuellement, utilisant des types PostgreSQL adaptés :
- `INTEGER`, `NUMERIC`, `TEXT`, `BIGINT`


In [26]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE movies (
            "movieId" INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            genres TEXT
        );

        CREATE TABLE ratings (
            "userId" INTEGER,
            "movieId" INTEGER,
            rating NUMERIC(2,1),
            "timestamp" BIGINT,
            PRIMARY KEY ("userId", "movieId", "timestamp")
        );

        CREATE TABLE tags (
            "userId" INTEGER,
            "movieId" INTEGER,
            tag TEXT,
            "timestamp" BIGINT,
            PRIMARY KEY ("userId", "movieId", "timestamp")
        );

        CREATE TABLE links (
            "movieId" INTEGER PRIMARY KEY,
            imdbId TEXT,
            tmdbId TEXT
        );
    """))
print("✅ Tables PostgreSQL recréées avec typage explicite.")


✅ Tables PostgreSQL recréées avec typage explicite.


In [27]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS movies CASCADE;

        CREATE TABLE movies (
            "movieId" INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            genres TEXT,
            main_genre TEXT,
            year INTEGER
        );
    """))
print("✅ Table `movies` recréée avec les colonnes enrichies.")


✅ Table `movies` recréée avec les colonnes enrichies.


In [28]:
movies = pd.read_csv('../data/clean/movies.csv')
movies.to_sql('movies', con=engine, index=False, if_exists='append')


742

In [29]:
from sqlalchemy import text

# Dictionnaire des DataFrames à insérer
dataframes = {
    'movies': pd.read_csv('../data/clean/movies.csv'),
    'ratings': pd.read_csv('../data/clean/ratings.csv'),
    'tags': pd.read_csv('../data/clean/tags.csv'),
    'links': pd.read_csv('../data/clean/links.csv')
}

# Vider les tables (dans l'ordre inverse des dépendances)
tables = ['ratings', 'tags', 'links', 'movies']
with engine.begin() as conn:
    for table in tables:
        conn.execute(text(f"TRUNCATE TABLE {table} RESTART IDENTITY CASCADE;"))
        print(f"🧹 Table `{table}` vidée.")

with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS ratings CASCADE;
        DROP TABLE IF EXISTS tags CASCADE;
        DROP TABLE IF EXISTS links CASCADE;
        DROP TABLE IF EXISTS movies CASCADE;

        CREATE TABLE movies (
            "movieId" INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            genres TEXT,
            main_genre TEXT,
            year INTEGER
        );

        CREATE TABLE ratings (
            "userId" INTEGER,
            "movieId" INTEGER,
            rating NUMERIC(2,1),
            "timestamp" BIGINT,
            date TIMESTAMP,
            year INTEGER,
            PRIMARY KEY ("userId", "movieId", "timestamp")
        );

        CREATE TABLE tags (
            "userId" INTEGER,
            "movieId" INTEGER,
            tag TEXT,
            "timestamp" BIGINT,
            PRIMARY KEY ("userId", "movieId", "timestamp")
        );

        CREATE TABLE links (
            "movieId" INTEGER PRIMARY KEY,
            "imdbId" INTEGER,
            "tmdbId" INTEGER
        );
    """))
print("✅ Tables recréées avec colonnes enrichies.")


# Réinsérer les données
for name, df in dataframes.items():
    df.to_sql(name, con=engine, index=False, if_exists='append')
    print(f"✅ Données insérées dans la table `{name}`.")


🧹 Table `ratings` vidée.
🧹 Table `tags` vidée.
🧹 Table `links` vidée.
🧹 Table `movies` vidée.
✅ Tables recréées avec colonnes enrichies.
✅ Données insérées dans la table `movies`.
✅ Données insérées dans la table `ratings`.
✅ Données insérées dans la table `tags`.
✅ Données insérées dans la table `links`.


📊 Contrôle du nombre de lignes dans chaque table
python
Copier


In [30]:
# Vérification des données insérées

# Tables à vérifier
tables = ['movies', 'ratings', 'tags', 'links']

# Connexion et requêtes
with engine.connect() as conn:
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table};"))
        count = result.scalar()
        print(f"📦 {table} → {count:,} lignes")


📦 movies → 9,742 lignes
📦 ratings → 100,836 lignes
📦 tags → 3,423 lignes
📦 links → 9,742 lignes


In [31]:
# Requête SQL pour voir toutes les contraintes des tables
query = """
SELECT
    tc.table_name,
    tc.constraint_type,
    tc.constraint_name,
    kcu.column_name,
    ccu.table_name AS foreign_table,
    ccu.column_name AS foreign_column
FROM
    information_schema.table_constraints AS tc
LEFT JOIN
    information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
LEFT JOIN
    information_schema.constraint_column_usage AS ccu
    ON ccu.constraint_name = tc.constraint_name
WHERE
    tc.table_schema = 'public'
ORDER BY
    tc.table_name, tc.constraint_type;
"""

pd.read_sql_query(query, engine)


,table_name,constraint_type,constraint_name,column_name,foreign_table,foreign_column
0,links,CHECK,2200_17069_1_not_null,None,None,None
1,links,PRIMARY KEY,links_pkey,movieId,links,movieId
2,movies,CHECK,2200_17050_2_not_null,None,None,None
3,movies,CHECK,2200_17050_1_not_null,None,None,None
4,movies,PRIMARY KEY,movies_pkey,movieId,movies,movieId
5,ratings,CHECK,2200_17057_4_not_null,None,None,None
6,ratings,CHECK,2200_17057_2_not_null,None,None,None
7,ratings,CHECK,2200_17057_1_not_null,None,None,None
8,ratings,PRIMARY KEY,ratings_pkey,timestamp,ratings,timestamp
9,ratings,PRIMARY KEY,ratings_pkey,timestamp,ratings,userId


In [32]:


with engine.begin() as conn:
    # 🔹 Supprimer les anciennes clés primaires (si elles existent)
    try:
        conn.execute(text("ALTER TABLE ratings DROP CONSTRAINT IF EXISTS ratings_pkey;"))
        conn.execute(text("ALTER TABLE tags DROP CONSTRAINT IF EXISTS tags_pkey;"))
        conn.execute(text("ALTER TABLE links DROP CONSTRAINT IF EXISTS links_pkey;"))
        conn.execute(text("ALTER TABLE movies DROP CONSTRAINT IF EXISTS movies_pkey;"))
        print("✅ Anciennes contraintes supprimées.")
    except Exception as e:
        print("⚠️ Erreur suppression : ", e)

    # 🔹 Créer les clés primaires propres
    try:
        conn.execute(text("""
            ALTER TABLE ratings
            ADD CONSTRAINT pk_ratings PRIMARY KEY ("userId", "movieId", "timestamp");
        """))
        conn.execute(text("""
            ALTER TABLE tags
            ADD CONSTRAINT pk_tags PRIMARY KEY ("userId", "movieId", "timestamp");
        """))
        conn.execute(text("""
            ALTER TABLE links
            ADD CONSTRAINT pk_links PRIMARY KEY ("movieId");
        """))
        conn.execute(text("""
            ALTER TABLE movies
            ADD CONSTRAINT pk_movies PRIMARY KEY ("movieId");
        """))
        print("✅ Nouvelles clés primaires créées avec succès.")
    except Exception as e:
        print("❌ Erreur lors de la création des clés primaires :", e)


✅ Anciennes contraintes supprimées.
✅ Nouvelles clés primaires créées avec succès.


In [33]:
# 🧱 Ajout des clés étrangères après la création des tables
from sqlalchemy import text

with engine.begin() as conn:
    try:
        # Clé étrangère ratings → movies
        conn.execute(text("""
            ALTER TABLE ratings
            ADD CONSTRAINT fk_ratings_movie
            FOREIGN KEY ("movieId") REFERENCES movies("movieId");
        """))

        # Clé étrangère tags → movies
        conn.execute(text("""
            ALTER TABLE tags
            ADD CONSTRAINT fk_tags_movie
            FOREIGN KEY ("movieId") REFERENCES movies("movieId");
        """))

        # Clé étrangère links → movies
        conn.execute(text("""
            ALTER TABLE links
            ADD CONSTRAINT fk_links_movie
            FOREIGN KEY ("movieId") REFERENCES movies("movieId");
        """))

        print("✅ Clés étrangères ajoutées avec succès.")
    except Exception as e:
        print("❌ Erreur lors de l’ajout des clés étrangères :", e)


✅ Clés étrangères ajoutées avec succès.


In [34]:
# Vérification d’une jointure simple
query = """
SELECT r."movieId", m.title, r.rating
FROM ratings r
JOIN movies m ON r."movieId" = m."movieId"
LIMIT 5;
"""

pd.read_sql_query(query, engine)


,movieId,title,rating
0,1,Toy Story (1995),4.0
1,3,Grumpier Old Men (1995),4.0
2,6,Heat (1995),4.0
3,47,Seven (a.k.a. Se7en) (1995),5.0
4,50,"Usual Suspects, The (1995)",5.0


In [38]:
# 🛠️ Requête pour visualiser toutes les contraintes définies dans PostgreSQL

query = """
SELECT 
    tc.table_name,
    tc.constraint_type,
    tc.constraint_name,
    kcu.column_name,
    ccu.table_name AS foreign_table,
    ccu.column_name AS foreign_column
FROM information_schema.table_constraints AS tc
LEFT JOIN information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_name = kcu.table_name
LEFT JOIN information_schema.constraint_column_usage AS ccu
    ON tc.constraint_name = ccu.constraint_name
WHERE tc.table_schema = 'public'
ORDER BY tc.table_name, tc.constraint_type;
"""

# Affichage du résultat sous forme de DataFrame
pd.read_sql_query(query, engine)


,table_name,constraint_type,constraint_name,column_name,foreign_table,foreign_column
0,links,CHECK,2200_17069_1_not_null,None,None,None
1,links,FOREIGN KEY,fk_links_movie,movieId,movies,movieId
2,links,PRIMARY KEY,pk_links,movieId,links,movieId
3,movies,CHECK,2200_17050_1_not_null,None,None,None
4,movies,CHECK,2200_17050_2_not_null,None,None,None
5,movies,PRIMARY KEY,pk_movies,movieId,movies,movieId
6,ratings,CHECK,2200_17057_1_not_null,None,None,None
7,ratings,CHECK,2200_17057_2_not_null,None,None,None
8,ratings,CHECK,2200_17057_4_not_null,None,None,None
9,ratings,FOREIGN KEY,fk_ratings_movie,movieId,movies,movieId


## ✅ Conclusion

Toutes les tables ont été insérées et structurées avec succès dans PostgreSQL :

- Données nettoyées et prêtes à l’analyse
- Contraintes relationnelles assurées
- Structure stable pour l’analyse SQL avancée (phase suivante)

Prochaine étape → `03_analyse_sql.ipynb` : extraction d'insights métier.
